# Weather Dataset Cleaning & Feature Engineering

This notebook prepares raw weather data for integration with transit performance data.  
The goal is to standardize formats, improve data quality, and create analytical features for later modeling.


## Data Source

This dataset was extracted from the Open-Meteo Historical Weather API.

It contains daily weather observations for Winnipeg for the year 2023, including temperature, wind speed, precipitation, and weather condition codes.


In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv('../../data_raw/weather/weather_raw.csv')
df

,date,weather_code,temperature_2m_max,precipitation_sum,temperature_2m_mean,temperature_2m_min,visibility_mean,wind_speed_10m_mean
0,2023-01-01 00:00:00+00:00,71.0,-6.30,0.4,-7.293750,-8.00,NaN,5.050155
1,2023-01-02 00:00:00+00:00,3.0,-7.00,0.0,-9.414583,-13.00,NaN,11.658464
2,2023-01-03 00:00:00+00:00,3.0,-5.00,0.0,-7.262500,-8.85,NaN,7.522199
3,2023-01-04 00:00:00+00:00,3.0,-7.65,0.0,-9.227084,-11.35,NaN,5.925349
4,2023-01-05 00:00:00+00:00,3.0,-7.15,0.0,-8.983334,-12.60,NaN,6.790324
...,...,...,...,...,...,...,...,...
360,2023-12-27 00:00:00+00:00,1.0,1.20,0.0,-4.641667,-8.75,NaN,4.848445
361,2023-12-28 00:00:00+00:00,1.0,0.05,0.0,-4.825000,-8.05,NaN,7.696402
362,2023-12-29 00:00:00+00:00,3.0,0.70,0.0,-3.793750,-6.45,NaN,13.218433
363,2023-12-30 00:00:00+00:00,73.0,-3.90,1.7,-5.414583,-8.65,NaN,20.797247


## Data Cleaning

Several transformations were applied to improve data consistency and usability:

- Removed the `visibility_mean` column due to missing values
- Converted the `date` column to a date-only format (removed timestamp)
- Converted `weather_code` from float to integer to correctly represent categorical weather conditions
- Renamed columns using a consistent and readable `snake_case` naming convention
- Rounded numerical averages (mean temperature and mean wind speed) to two decimal places for clarity


In [45]:
df = df.drop(columns=['visibility_mean'])

In [46]:
df['date'] = df['date'].apply(
    lambda timestamp: timestamp[:10].strip()
)
df.head(3)

,date,weather_code,temperature_2m_max,precipitation_sum,temperature_2m_mean,temperature_2m_min,wind_speed_10m_mean
0,2023-01-01,71.0,-6.3,0.4,-7.293750,-8.00,5.050155
1,2023-01-02,3.0,-7.0,0.0,-9.414583,-13.00,11.658464
2,2023-01-03,3.0,-5.0,0.0,-7.262500,-8.85,7.522199


In [47]:
df['weather_code'] = df['weather_code'].apply(int)
df.head(3)

,date,weather_code,temperature_2m_max,precipitation_sum,temperature_2m_mean,temperature_2m_min,wind_speed_10m_mean
0,2023-01-01,71,-6.3,0.4,-7.293750,-8.00,5.050155
1,2023-01-02,3,-7.0,0.0,-9.414583,-13.00,11.658464
2,2023-01-03,3,-5.0,0.0,-7.262500,-8.85,7.522199


In [50]:
df = df.rename(columns={
    'temperature_2m_max': 'max_temperature',
    'precipitation_sum': 'sum_precipitation',
    'temperature_2m_mean': 'mean_temperature',
    'temperature_2m_min': 'min_temperature',
    'wind_speed_10m_mean': 'mean_wind_speed'
})
df.columns.tolist()

['date',
 'weather_code',
 'min_temperature',
 'max_temperature',
 'mean_temperature',
 'sum_precipitation',
 'mean_wind_speed']

In [52]:
df['mean_temperature'] = [round(temp, 2) for temp in df['mean_temperature']]
df.head(3)

,date,weather_code,min_temperature,max_temperature,mean_temperature,sum_precipitation,mean_wind_speed
0,2023-01-01,71,-8.00,-6.3,-7.29,0.4,5.050155
1,2023-01-02,3,-13.00,-7.0,-9.41,0.0,11.658464
2,2023-01-03,3,-8.85,-5.0,-7.26,0.0,7.522199


In [53]:
df['mean_wind_speed'] = [round(ws, 2) for ws in df['mean_wind_speed']]
df.head(3)

,date,weather_code,min_temperature,max_temperature,mean_temperature,sum_precipitation,mean_wind_speed
0,2023-01-01,71,-8.00,-6.3,-7.29,0.4,5.05
1,2023-01-02,3,-13.00,-7.0,-9.41,0.0,11.66
2,2023-01-03,3,-8.85,-5.0,-7.26,0.0,7.52


## Feature Engineering

New feature was created to support later analysis and integration with transit performance data:

- **is_rainy** → Boolean flag indicating whether precipitation occurred on a given day  
  (based on `sum_precipitation > 0`)

This derived field help simplify analysis of how weather conditions may affect public transportation performance.


In [55]:
df['is_rainy'] = df['sum_precipitation'].apply(
    lambda p: 1 if p
    else 0
)
df

,date,weather_code,min_temperature,max_temperature,mean_temperature,sum_precipitation,mean_wind_speed,is_rainy
0,2023-01-01,71,-8.00,-6.30,-7.29,0.4,5.05,True
1,2023-01-02,3,-13.00,-7.00,-9.41,0.0,11.66,False
2,2023-01-03,3,-8.85,-5.00,-7.26,0.0,7.52,False
3,2023-01-04,3,-11.35,-7.65,-9.23,0.0,5.93,False
4,2023-01-05,3,-12.60,-7.15,-8.98,0.0,6.79,False
...,...,...,...,...,...,...,...,...
360,2023-12-27,1,-8.75,1.20,-4.64,0.0,4.85,False
361,2023-12-28,1,-8.05,0.05,-4.83,0.0,7.70,False
362,2023-12-29,3,-6.45,0.70,-3.79,0.0,13.22,False
363,2023-12-30,73,-8.65,-3.90,-5.41,1.7,20.80,True


## Reorder Columns

In [56]:
new_column_order = [
    'date', 'weather_code',
    'min_temperature', 'max_temperature',
    'mean_temperature', 'mean_wind_speed',
    'sum_precipitation', 'is_rainy'
]
df = df[new_column_order]
df.head(1)

,date,weather_code,min_temperature,max_temperature,mean_temperature,mean_wind_speed,sum_precipitation,is_rainy
0,2023-01-01,71,-8.0,-6.3,-7.29,5.05,0.4,True


## Output Dataset

The cleaned dataset now represents a daily weather dimension table, structured for future integration with transit operational data.

Key columns include:

- Date
- Weather condition code
- Minimum, maximum, and mean temperature
- Mean wind speed
- Total daily precipitation
- Rain indicator flag


In [57]:
df.to_csv('weather_clean.csv', index=False)